# Save temperature on pressure fields

To convert PM2.5 to a concentration rather than a mass mixing ratio we must calculate air density from pressure and temperature. 

This script is an example of processing CESM2 temperature data and may be different depending on where you download/access your data from.

In [ ]:
import os
import re
import warnings
import xarray as xr
from utils.utils import get_scenario_config, load_file_list, minus_one_month

In [ ]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]

FILE_DIR = f"/glade/work/awells/air_quality/{model}/temp/file_paths/"
SAVE_DIR = f"/glade/work/awells/air_quality/{model}/temp/temp_pres/"

In [ ]:
def load_file(f):
    if not os.path.exists(f):
        raise ValueError(f"Missing: {f}")

    # Find variable
    pattern = re.compile(r"cam\.h0\.([^.]+)\.")
    m = pattern.search(f)
    varname = m.group(1)

    ds = xr.open_dataset(f)
    da = ds[varname]
    return da

In [ ]:
for ens_num in ensemble_members:
    print(f"Processing {scenario}, ensemble {ens_num:02d}")
    # Load all file lists
    file_list = load_file_list(FILE_DIR, f"file_list_T_{scenario}_{ens_num:02d}.json")

    das = xr.open_mfdataset(file_list)["T"]

    # If the first month is February (2) then apply month fixer
    first_month = das.time.dt.month[0]
    if first_month == 2:
        print("Adjusting month indexing")
        new_time = [minus_one_month(t) for t in das["time"].values]
        das = das.assign_coords(time=new_time)
    # If the first month is January (1) don't apply month fixer
    elif first_month == 1:
        print("No month index adjusting needed")
    else:
        warnings.warn(f"First month: {first_month}, check dates in file")

    first_year = das.time.dt.year[0].item()
    last_year = das.time.dt.year[-1].item()

    # Saving can take a while...
    out_file = f"T_{model}_{scenario}_{ens_num:02d}_{first_year}-{last_year}.nc"
    out_path = os.path.join(SAVE_DIR, out_file)
    print(f"Saving to {out_path}")
    das.to_netcdf(out_path)

print("All processing complete.")